In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import mne
import torch
from pathlib import Path
from dn3_ext import BENDRClassification

data_dir = Path("data")
edf_path_demo = data_dir / "our_demo" / "test_from_20s.edf"

def load_modules(mdl: BENDRClassification, enc_weights: Path, cont_weights: Path) -> None:
    device = "cpu" # TODO if torch.cuda.is_available, etc

    enc_sd = torch.load(str(enc_weights), map_location=device)
    mdl.encoder.load_state_dict(enc_sd, strict=False)

    ctx_sd = torch.load(str(cont_weights), map_location=device)
    mdl.contextualizer.load_state_dict(ctx_sd, strict=False)


In [ ]:
raw_demo = mne.io.read_raw_edf(edf_path_demo, preload=True, verbose=False)
raw_demo.drop_channels(['ECG  ECG', 'EEG OZ-A1']) # well, not exactly meaningfull channels
weights_dir = Path("pretrained_weights")

# pass raw_demo through BENDR

In [ ]:
arr: np.ndarray = raw_demo.get_data()  # shape: channels x samples
x = torch.from_numpy(arr[np.newaxis, ...]).float()  # shape: batch x channels x samples

tueg_model = BENDRClassification(targets=2, samples=arr.shape[1], channels=arr.shape[0])
# model.load_pretrained_modules(weights_dir / "encoder.pt",
#                               weights_dir / "contextualizer.pt",
#                               strict=False)
load_modules(tueg_model, weights_dir / "encoder.pt", weights_dir / "contextualizer.pt")

# make sure `mask_replacement` is not trainable
if hasattr(tueg_model.contextualizer, "mask_replacement"):
    tueg_model.contextualizer.mask_replacement.requires_grad = False

tueg_model.eval()
with torch.no_grad():
    feats = tueg_model.features_forward(x)

print("feature shape:", feats.shape)
print("first values:", feats.flatten()[:10])

# Load an example from TUEV

In [ ]:
mne.io.read_raw_edf(data_dir / "tueg_demo" / "aaaaaeri_00000001.edf").ch_names

In [ ]:
raw_demo.ch_names

In [ ]:
from dn3.dn3.configuratron import ExperimentConfig
from dn3.dn3.transforms.instance import To1020

tueg_config_path = Path("configs") / "tueg_local_demo.yml"
tueg_experiment_config = ExperimentConfig(str(tueg_config_path))
tueg_config = tueg_experiment_config.datasets['tueg']
tueg_dataset = tueg_config.auto_construct_dataset()
tueg_dataset.add_transform(To1020())

In [ ]:
from dn3_ext import BENDRClassification

tueg_model = BENDRClassification.from_dataset(tueg_dataset)
load_modules(tueg_model, Path(tueg_experiment_config.encoder_weights), Path(tueg_experiment_config.context_weights)) # TODO freeze encoder

# Load our data using D3N tools

In [ ]:
segment_length = 3.0  # seconds

# Number of complete 3-second segments
duration = raw_demo.times[-1]
n_segments = int(duration // segment_length)

# Random labels
rng = np.random.default_rng(42)
labels = rng.choice(["b", "nb"], size=n_segments)  # blink / non-blink

# Annotation start times
onsets = np.arange(n_segments) * segment_length

# random subset
rand_mask = np.random.choice(2, n_segments, p=[4/5, 1/5]) == 1
sub_labels = labels[rand_mask]
sub_onsets = onsets[rand_mask]
durations = np.full(len(sub_labels), segment_length)
annotations = mne.Annotations(
    onset=sub_onsets,
    duration=durations,
    description=sub_labels.tolist()
)

raw_copy = raw_demo.copy()
raw_copy.set_annotations(annotations)

mne.export.export_raw(
    edf_path_demo.parent.parent / f"{edf_path_demo.parent.stem}_with_labels" / f"{edf_path_demo.stem}.edf",
    raw_copy,
    fmt="edf",
    overwrite=True,
)

In [ ]:
from dn3.dn3.configuratron import ExperimentConfig
from dn3.dn3.transforms.instance import To1020
from dn3_ext import BENDRClassification

ours_config_path = Path("configs") / "ours_local_demo.yml"
ours_experiment_config = ExperimentConfig(str(ours_config_path))
ours_data_config = ours_experiment_config.datasets['demo']
ours_demo_dataset = ours_data_config.auto_construct_dataset()
ours_demo_dataset.add_transform(To1020())

model_ours = BENDRClassification.from_dataset(ours_demo_dataset)
load_modules(model_ours, Path(ours_experiment_config.encoder_weights), Path(ours_experiment_config.context_weights))

In [ ]:
ours_demo_dataset.get_targets()[:5]

In [ ]:
import torch
from dn3.dn3.trainable.processes import StandardClassification

metrics_retain = "Accuracy"
process = StandardClassification(model_ours,metrics = [metrics_retain]
                                 )
process.set_optimizer(torch.optim.AdamW(process.parameters(), ours_data_config.lr, weight_decay=.01))
process.fit(training_dataset=ours_demo_dataset, retain_best = metrics_retain)

# for train, val, test in ours_demo_dataset.lmso(1,
#                                                test_splits=[["copy1"]],
#                                                validation_splits=[["copy2"]]
#                                                ):
#     process = StandardClassification(model_ours,
#                                     #   metrics=[metrics_retain]
#                                       )
#     process.set_optimizer(torch.optim.AdamW(process.parameters(), ours_data_config.lr, weight_decay=.01))
#     process.fit(training_dataset=train, validation_dataset=val, 
#                 # retain_best=metrics_retain
#                 )